# Sentiment-Analyse von TikTok-Kommentaren

Dieses Notebook führt eine Sentiment-Analyse (Stimmungsanalyse) für Kommentare durch, die von einer TikTok-Video-Seite mit einem Selenium-Skript gesammelt und in einer JSON-Datei gespeichert wurden.

**Ziel:** Die vorherrschende Stimmung (positiv, negativ, neutral) in den Kommentaren zu einem bestimmten TikTok-Video zu ermitteln.

**Verwendetes Modell:** Wir nutzen das multilinguale Sentiment-Analyse-Modell `tabularisai/multilingual-sentiment-analysis` von Hugging Face. Dieses Modell kann Kommentare in verschiedenen Sprachen analysieren, ist aber möglicherweise weniger präzise für eine spezifische Sprache als ein dediziertes Modell für diese Sprache.

**Wichtig:** Dieses Notebook dient zu Demonstrations- und Lernzwecken. Die Ergebnisse einer automatisierten Sentiment-Analyse sollten stets mit Vorsicht interpretiert werden, da Kontext, Ironie und Sarkasmus oft schwer zu erfassen sind.

## 1. Vorbereitung: Bibliotheken installieren und JSON-Datei hochladen

Zuerst installieren wir die notwendigen Python-Bibliotheken. Anschließend müssen Sie die JSON-Datei, die von Ihrem TikTok-Scraper generiert wurde (z.B. `tiktok_data_VIDEOID.json`), in das Arbeitsverzeichnis dieses Colab-Notebooks hochladen.

**Anleitung zum Hochladen:**
1. Klicken Sie auf das Ordner-Symbol in der linken Seitenleiste von Colab.
2. Klicken Sie auf das "Hochladen"-Symbol (Pfeil nach oben) und wählen Sie Ihre JSON-Datei aus.

In [ ]:
# Installiere die 'transformers' Bibliothek für die Sentiment-Analyse,
# 'pandas' für die Datenverarbeitung und 'matplotlib' sowie 'seaborn' für Visualisierungen.
!pip install transformers[sentencepiece] pandas matplotlib seaborn

In [ ]:
# Importiere benötigte Bibliotheken
import json
import pandas as pd
from transformers import pipeline
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re # Für die Like-Konvertierung

print('Bibliotheken erfolgreich importiert.')

## 2. JSON-Datei laden und Kommentare extrahieren

Geben Sie hier den Namen Ihrer hochgeladenen JSON-Datei an.

In [ ]:
# BITTE DEN DATEINAMEN ANPASSEN, falls Ihre Datei anders heißt!
json_filename = 'tiktok_data_7500353799500926230.json' # Beispielname, bitte an Ihre Datei anpassen

try:
    with open(json_filename, 'r', encoding='utf-8') as f:
        video_data_raw = json.load(f)
    print(f'JSON-Datei "{json_filename}" erfolgreich geladen.')
    
    # Kommentare extrahieren und in einen Pandas DataFrame umwandeln
    if 'comments' in video_data_raw and isinstance(video_data_raw['comments'], list) and video_data_raw['comments']:
        df_comments = pd.DataFrame(video_data_raw['comments'])
        print(f'Anzahl der Kommentare: {len(df_comments)}')
        print(f'Spalten im Kommentar-DataFrame: {df_comments.columns.tolist()}')
        print("\nErste paar Kommentare zur Überprüfung:")
        from IPython.display import display # Für schönere Ausgabe in Colab
        display(df_comments.head())
    else:
        print("Keine Kommentare in der JSON-Datei gefunden oder die Kommentarliste ist leer.")
        df_comments = pd.DataFrame() # Leerer DataFrame, falls keine Kommentare
except FileNotFoundError:
    print(f"FEHLER: Die Datei '{json_filename}' wurde nicht gefunden. Bitte laden Sie sie hoch und überprüfen Sie den Namen.")
except json.JSONDecodeError:
    print(f"FEHLER: Die Datei '{json_filename}' ist keine valide JSON-Datei. Bitte überprüfen Sie den Inhalt.")
except Exception as e:
    print(f"Ein Fehler ist beim Laden oder Verarbeiten der JSON-Datei aufgetreten: {e}")

## 3. Datenbereinigung und -vorbereitung

Wir bereinigen die Kommentartexte leicht und konvertieren die Like-Zahlen in numerische Werte, falls vorhanden.

In [ ]:
def convert_like_count(like_str):
    """Konvertiert Like-Angaben wie '1.2K' oder '1M' in Zahlen."""
    if isinstance(like_str, (int, float)):
        return int(like_str)
    if pd.isna(like_str) or not isinstance(like_str, str):
        return 0
    
    like_str_cleaned = like_str.lower().strip().replace(',', '.') # Ersetze Komma durch Punkt für Floats
    if 'k' in like_str_cleaned:
        return int(float(like_str_cleaned.replace('k', '')) * 1000)
    elif 'm' in like_str_cleaned:
        return int(float(like_str_cleaned.replace('m', '')) * 1000000)
    elif like_str_cleaned.replace('.', '', 1).isdigit(): # Erlaube einen Punkt für Dezimalzahlen (obwohl bei Likes selten)
        return int(float(like_str_cleaned))
    return 0 # Fallback für unbekannte Formate oder 'N/A'

if 'df_comments' in locals() and not df_comments.empty:
    # Fehlende Texte durch leere Strings ersetzen (wichtig für die Pipeline)
    df_comments['text'] = df_comments['text'].fillna('').astype(str)
    
    # Like-Zahlen konvertieren
    if 'likes' in df_comments.columns:
        df_comments['likes_numeric'] = df_comments['likes'].apply(convert_like_count)
        print("\nLikes konvertiert (Beispiel):")
        from IPython.display import display # Import display for better Colab output
        display(df_comments[['likes', 'likes_numeric']].head())
    else:
        print("Spalte 'likes' nicht im DataFrame gefunden. Setze 'likes_numeric' auf 0.")
        df_comments['likes_numeric'] = 0
        
    # Entferne Kommentare ohne Text, falls welche durchgerutscht sind
    initial_comment_count = len(df_comments)
    df_comments = df_comments[df_comments['text'].str.strip() != '']
    print(f"Anzahl Kommentare vor Entfernung leerer Texte: {initial_comment_count}")
    print(f"Anzahl Kommentare nach Entfernung leerer Texte: {len(df_comments)}")
else:
    print("Kommentar-DataFrame ist leer oder nicht geladen. Überspringe Datenbereinigung.")

## 4. Initialisierung der Sentiment-Analyse Pipeline

Wir laden das vortrainierte multilinguale Modell. Dies kann beim ersten Mal einige Minuten dauern, da das Modell (ca. 1-2 GB) heruntergeladen wird.

In [ ]:
# Initialisiere die Sentiment-Analyse Pipeline
print('Initialisiere multilinguale Sentiment-Analyse Pipeline... Dies kann einen Moment dauern.')
sentiment_analyzer = None # Initialize to prevent error if try block fails
try:
    sentiment_analyzer = pipeline(
        task='sentiment-analysis', 
        model='tabularisai/multilingual-sentiment-analysis', 
        # device=0 # Für GPU-Nutzung entkommentieren (falls CUDA verfügbar ist)
    )
    print('Sentiment-Analyse Pipeline erfolgreich initialisiert.')
except Exception as e:
    print(f"Fehler bei der Initialisierung der Pipeline: {e}")
    print("Stellen Sie sicher, dass eine Internetverbindung besteht und die 'transformers'-Bibliothek korrekt installiert ist.")

## 5. Sentiment-Analyse der Kommentare durchführen

Nun wenden wir das initialisierte Modell auf jeden Kommentartext an. Da dies für viele Kommentare dauern kann, geben wir alle 50 Kommentare eine Statusmeldung aus.

In [ ]:
def analyze_sentiment_for_comment(text_to_analyze):
    """Analysiert den Sentiment eines einzelnen Kommentartextes."""
    if not text_to_analyze or pd.isna(text_to_analyze):
        return {'label': 'NO_TEXT', 'score': 0.0}
    try:
        max_char_length = 400 
        truncated_text = text_to_analyze[:max_char_length]
        
        result = sentiment_analyzer(truncated_text)
        if result and isinstance(result, list):
            return result[0] 
        else:
            return {'label': 'ERROR_PARSING_RESULT', 'score': 0.0}
    except Exception as e:
        # print(f"Fehler bei der Analyse von Text: '{str(text_to_analyze)[:50]}...': {e}") # Für Debugging
        return {'label': 'ERROR_ANALYSIS', 'score': 0.0}

if 'df_comments' in locals() and not df_comments.empty and 'sentiment_analyzer' in locals() and sentiment_analyzer is not None:
    print(f"Führe Sentiment-Analyse für {len(df_comments)} Kommentare durch...")
    
    sentiment_labels = []
    sentiment_scores = []
    
    start_time_analysis = time.time()
    
    for index, row in df_comments.iterrows(): 
        comment_text = row['text']
        
        if not comment_text.strip():
            sentiment_result = {'label': 'NO_TEXT', 'score': 0.0}
        else:
            sentiment_result = analyze_sentiment_for_comment(comment_text)
        
        sentiment_labels.append(sentiment_result['label'])
        sentiment_scores.append(sentiment_result.get('score', 0.0))
        
        if (index + 1) % 50 == 0 or (index + 1) == len(df_comments):
            elapsed_time = time.time() - start_time_analysis
            print(f"--- {index + 1}/{len(df_comments)} Kommentare verarbeitet in {elapsed_time:.2f} Sekunden ---")

    df_comments['sentiment_label'] = sentiment_labels
    df_comments['sentiment_score'] = sentiment_scores
    
    end_time_analysis = time.time()
    print(f'\nSentiment-Analyse für {len(df_comments)} Kommentare abgeschlossen in {(end_time_analysis - start_time_analysis):.2f} Sekunden.')
    
    print("\nDataFrame mit Sentiment-Ergebnissen (erste Zeilen):")
    from IPython.display import display # Import display for better Colab output
    display(df_comments[['author', 'text', 'likes_numeric', 'sentiment_label', 'sentiment_score']].head())
    
    base_name = json_filename.rsplit('.', 1)[0]
    output_filename_csv = f"{base_name}_mit_sentiment.csv"
    
    try:
        df_comments.to_csv(output_filename_csv, index=False, encoding='utf-8-sig')
        print(f"\nDaten mit Sentiment-Analyse gespeichert in CSV: {output_filename_csv}")
    except Exception as e:
        print(f"Fehler beim Speichern der Ergebnisse: {e}")

else:
    print("Voraussetzungen für die Analyse (DataFrame, Pipeline) nicht erfüllt. Analyse wird übersprungen.")

## 6. Auswertung und Visualisierung der Ergebnisse

Nachdem die Sentiment-Analyse durchgeführt wurde, können wir uns die Verteilung der Sentiments ansehen und einige einfache Auswertungen vornehmen.

In [ ]:
if 'df_comments' in locals() and not df_comments.empty and 'sentiment_label' in df_comments.columns:
    print("\n--- Auswertung der Sentiment-Ergebnisse ---")
    
    # 1. Verteilung der Sentiment-Labels
    sentiment_counts = df_comments['sentiment_label'].value_counts()
    sentiment_percentages = df_comments['sentiment_label'].value_counts(normalize=True) * 100
    
    print("\nVerteilung der Sentiment-Labels:")
    from IPython.display import display # Import display for better Colab output
    display(sentiment_counts)
    print("\nProzentuale Verteilung der Sentiment-Labels:")
    display(sentiment_percentages.round(2).astype(str) + ' %')
    
    # 2. Visualisierung der Sentiment-Verteilung
    plt.figure(figsize=(10, 7))
    sns.set_style("whitegrid")
    ax = sns.countplot(data=df_comments, x='sentiment_label', order=sentiment_counts.index, palette='viridis')
    plt.title('Verteilung der Sentiment-Labels in den Kommentaren', fontsize=16, pad=20)
    plt.xlabel('Sentiment Label', fontsize=13, labelpad=10)
    plt.ylabel('Anzahl der Kommentare', fontsize=13, labelpad=10)
    plt.xticks(rotation=0, fontsize=11)
    plt.yticks(fontsize=11)
    
    # Füge die Anzahl und Prozentzahl über den Balken hinzu
    total_comments = len(df_comments)
    for i, p in enumerate(ax.patches):
        height = p.get_height()
        ax.text(p.get_x() + p.get_width()/2.,
                height + (0.01 * total_comments), # Kleiner Offset über dem Balken
                f'{int(height)}\n({sentiment_percentages.iloc[i]:.1f}%)',
                ha="center", va="bottom", fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # 3. Durchschnittliche Konfidenz pro Label anzeigen
    if 'sentiment_score' in df_comments.columns:
        avg_confidence = df_comments.groupby('sentiment_label')['sentiment_score'].mean().sort_values(ascending=False)
        print("\nDurchschnittliche Konfidenz (Score) pro Sentiment-Label:")
        display(avg_confidence.round(4))
        
        plt.figure(figsize=(10, 6))
        avg_confidence.plot(kind='bar', color=sns.color_palette('coolwarm_r', len(avg_confidence)))
        plt.title('Durchschnittliche Konfidenz pro Sentiment-Label', fontsize=16, pad=20)
        plt.xlabel('Sentiment Label', fontsize=13, labelpad=10)
        plt.ylabel('Durchschnittlicher Score (Konfidenz)', fontsize=13, labelpad=10)
        plt.xticks(rotation=0, fontsize=11)
        plt.yticks(fontsize=11)
        plt.ylim(0, 1) 
        plt.grid(axis='y', linestyle='--')
        plt.tight_layout()
        plt.show()
        
    # 4. Beispiele für Kommentare pro Sentiment-Kategorie
    print("\nBeispiel-Kommentare pro Sentiment-Kategorie:")
    for label in sentiment_counts.index:
        print(f"\n--- {label.upper()} KOMMENTARE (bis zu 3 Beispiele) ---")
        # Stelle sicher, dass genügend Kommentare für das Sampling vorhanden sind
        sample_size = min(3, len(df_comments[df_comments['sentiment_label'] == label]))
        if sample_size > 0:
          sample_comments = df_comments[df_comments['sentiment_label'] == label]['text'].sample(sample_size).tolist()
          for i, comment_text in enumerate(sample_comments):
              print(f"{i+1}. {comment_text[:200]}{'...' if len(comment_text) > 200 else ''}")
        else:
            print("Keine Kommentare in dieser Kategorie für ein Sample verfügbar.")

else:
    print("DataFrame oder Sentiment-Spalten nicht vorhanden für die detaillierte Auswertung.")

## 7. Mögliche Weiterführungen und Limitationen

### Weiterführende Analysen:
- **Sentiment im Zeitverlauf:** Wenn Ihre Daten exakte Zeitstempel für Kommentare hätten (z.B. genaue Upload-Zeiten statt nur relative Angaben wie "Vor 2 T."), könnten Sie analysieren, wie sich das Sentiment über die Zeit entwickelt hat. Dies erfordert eine Anpassung des Scrapers, um präzisere Zeitdaten zu erfassen.
- **Sentiment in Relation zu Likes:** Untersuchen, ob Kommentare mit bestimmtem Sentiment tendenziell mehr (oder weniger) Likes erhalten. Die aktuelle Konvertierung der Like-Zahlen ist ein erster Schritt. Für eine tiefere Analyse ist die Qualität der gescrapten Like-Daten entscheidend.
- **Themenmodellierung (Topic Modeling):** Innerhalb der positiven/negativen/neutralen Kommentare könnten Sie versuchen, die Hauptthemen zu identifizieren (z.B. mit Techniken wie LDA oder BERTopic).
- **Wortwolken:** Erstellen Sie Wortwolken für jede Sentiment-Kategorie, um häufige und prägnante Begriffe zu visualisieren.
- **Vergleich zwischen Videos/Autoren:** Führen Sie diese Analyse für mehrere TikTok-Videos (ggf. von verschiedenen Autoren oder zu verschiedenen Themen) durch und vergleichen Sie die Sentiment-Verteilungen, um Muster oder Unterschiede in der Rezeption aufzudecken.
- **Analyse von Nutzerprofilen (falls relevant und ethisch vertretbar):** Wenn das Scraper-Skript auch Informationen über die kommentierenden Nutzer sammelt, könnten Zusammenhänge zwischen Nutzercharakteristika und dem geäußerten Sentiment untersucht werden (erfordert strenge Beachtung der DSGVO).

### Limitationen:
- **Genauigkeit des Modells:** Kein Sentiment-Analyse-Modell ist perfekt. Besonders bei subtilen Äußerungen, Ironie, Sarkasmus, kulturellen Referenzen oder sehr kontextabhängigen Aussagen kann es zu Fehlklassifikationen kommen.
- **Sprachliche Nuancen:** Emojis, Umgangssprache, Dialekte und Abkürzungen (die in TikTok-Kommentaren häufig sind) können die Analyse erschweren. Das multilinguale Modell versucht, dies breiter abzudecken, kann aber in spezifischen Sprachen weniger nuanciert sein.
- **Mehrdeutigkeit:** Manche Kommentare sind objektiv schwer einer einzigen Sentiment-Kategorie zuzuordnen.
- **Sampling Bias des Scrapings:** Die gescrapten Kommentare repräsentieren möglicherweise nicht alle Kommentare, die jemals zu dem Video gepostet wurden (z.B. wenn nicht alle durch Scrollen geladen werden konnten oder einige später vom Nutzer oder von TikTok gelöscht wurden).
- **Verarbeitung von 'N/A' oder '0' Likes:** Das Skript behandelt 'N/A' und '0' bei Likes als numerisch 0. Dies ist eine Vereinfachung. Eine genauere Erfassung der Likes im Scraper wäre wünschenswert, ist aber oft eine Herausforderung.

**Empfehlung:** Betrachten Sie die Ergebnisse dieser automatisierten Analyse als einen ersten, quantitativen Indikator. Für ein tiefergehendes Verständnis der Nuancen und des Kontexts ist es oft unerlässlich, diese Ergebnisse durch eine qualitative Analyse ausgewählter Kommentare zu ergänzen und kritisch zu hinterfragen.